In [ ]:
# %% [markdown]
# # Backprop Gradient & Learning Rate Visualization Experiment
# File: `backprop_gradient_visualization.ipynb`

# %%
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# %% [markdown]
# ## 1. MLP Implementation

# %%
class MLP:
    def __init__(self, input_dim: int, hidden_layers: int, hidden_layer_dim: int, output_dim: int, lr: float):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.hidden_layer_dim = hidden_layer_dim
        self.output_dim = output_dim
        self.lr = lr

        self.weight_matrix = []
        self.output_activations = []
        self.input = []
        self.gradients = []

        total_matrices = 1 + self.hidden_layers

        # He / Kaiming Normal Initialization for ReLU
        for i in range(total_matrices):
            if i == 0:
                scale = np.sqrt(2.0 / self.input_dim)
                self.weight_matrix.append(np.random.randn(self.hidden_layer_dim, self.input_dim + 1) * scale)
            elif i < total_matrices - 1:
                scale = np.sqrt(2.0 / self.hidden_layer_dim)
                self.weight_matrix.append(np.random.randn(self.hidden_layer_dim, self.hidden_layer_dim + 1) * scale)
            else:
                scale = np.sqrt(2.0 / self.hidden_layer_dim)
                self.weight_matrix.append(np.random.randn(self.output_dim, self.hidden_layer_dim + 1) * scale)

    def activation(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)  # ReLU

    def activation_derivative(self, x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

    def forward_propogate(self, X: np.ndarray) -> np.ndarray:
        self.input = []
        self.output_activations = []

        # Add bias feature to X -> Shape: (N, input_dim + 1)
        X_b = np.c_[X, np.ones((X.shape[0], 1))]
        A = X_b.T  # Shape: (input_dim + 1, N)
        self.output_activations.append(A)

        for i in range(len(self.weight_matrix)):
            Z = self.weight_matrix[i] @ A
            self.input.append(Z)

            if i < len(self.weight_matrix) - 1:
                A = self.activation(Z)
                A = np.r_[A, np.ones((1, A.shape[1]))]  # Add bias row
                self.output_activations.append(A)

        return Z.T  # Shape: (N, output_dim)

    def compute_loss(self, Y_pred: np.ndarray, Y: np.ndarray) -> float:
        # Mean Squared Error (MSE)
        return float(np.mean((Y_pred - Y) ** 2))

    def backward(self, X: np.ndarray, Y: np.ndarray) -> None:
        N = X.shape[0]
        self.gradients = [None] * len(self.weight_matrix)

        # Output layer forward output Z_out
        Z_out = self.input[-1]
        
        # Derivative of MSE Loss w.r.t Z_out
        dZ = 2 * (Z_out - Y.T) / N
        self.gradients[-1] = dZ @ self.output_activations[-1].T

        # Backpropagation through hidden layers
        for i in range(len(self.weight_matrix) - 2, -1, -1):
            W_nobias = self.weight_matrix[i + 1][:, :-1]
            dA = W_nobias.T @ dZ
            dZ = dA * self.activation_derivative(self.input[i])
            self.gradients[i] = dZ @ self.output_activations[i].T

    def update(self) -> None:
        for i in range(len(self.weight_matrix)):
            self.weight_matrix[i] -= self.lr * self.gradients[i]

# %% [markdown]
# ## 2. Generate Synthetic Dataset

# %%
# Synthetic non-linear dataset
X = np.random.uniform(-3, 3, size=(200, 2))
Y = np.sin(X[:, 0:1]) + 0.5 * X[:, 1:2] + np.random.normal(0, 0.1, size=(200, 1))

print(f"X shape: {X.shape}, Y shape: {Y.shape}")

# %% [markdown]
# ## 3. Experiment: Training Across Learning Rates

# %%
learning_rates = [0.0001, 0.001, 0.01, 0.1]
iterations = 500
loss_histories = {}

for lr in learning_rates:
    np.random.seed(42)  # Fixed initial weights across runs for fair comparison
    
    mlp = MLP(
        input_dim=2,
        hidden_layers=2,
        hidden_layer_dim=16,
        output_dim=1,
        lr=lr
    )
    
    history = []
    for it in range(iterations):
        # Forward pass
        Y_pred = mlp.forward_propogate(X)
        
        # Loss calculation
        loss = mlp.compute_loss(Y_pred, Y)
        history.append(loss)
        
        # Backward pass & update
        mlp.backward(X, Y)
        mlp.update()
        
    loss_histories[lr] = history

# %% [markdown]
# ## 4. Plot Loss vs Iteration

# %%
plt.figure(figsize=(10, 6))

for lr, history in loss_histories.items():
    plt.plot(range(1, iterations + 1), history, label=f"$\eta = {lr}$", linewidth=2)

plt.title("MLP Training Loss vs Iteration Across Learning Rates", fontsize=14, fontweight='bold')
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)
plt.yscale("log")  # Logarithmic scale highlights gradient performance differences clearly
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()